# 🧪 Professional A/B Testing Playbook — Enhanced Edition

## Comprehensive Statistical Testing Framework with Business Impact Analysis

This notebook provides a production-ready A/B testing framework with:
- **Advanced statistical methods**: Sequential testing, CUPED, Bayesian analysis, DiD
- **Reusable functions**: Sample size calculators, SRM checks, novelty detection
- **Interactive visualizations**: Plotly dashboards, decision trees, power heatmaps
- **Business impact analysis**: Revenue projections, cost-benefit analysis, sensitivity testing
- **Automated decision framework**: Winner declaration with guardrails

**Version**: 2.0.0 | **Last Updated**: January 2024

## 📋 1. Configuration & Setup

In [ ]:
# =============================================================================
# COMPREHENSIVE CONFIGURATION CELL
# =============================================================================

from __future__ import annotations
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Tuple, Optional, Union, Any, Literal
from datetime import datetime, timedelta
from enum import Enum
import json
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy import optimize
import math

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"

# Statistical modeling
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import tt_solve_power
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

@dataclass
class ExperimentConfig:
    """Master configuration for A/B test experiments"""
    
    # Experiment metadata
    experiment_name: str = "Landing Page Optimization"
    experiment_id: str = "EXP-2024-001"
    description: str = "Testing new landing page design vs control"
    hypothesis: str = "New design will increase conversion rate by 2%"
    
    # Dates
    start_date: datetime = field(default_factory=lambda: datetime(2024, 1, 1))
    end_date: datetime = field(default_factory=lambda: datetime(2024, 1, 31))
    
    # Statistical parameters
    alpha: float = 0.05  # Significance level
    power: float = 0.80  # Statistical power
    mde_relative: float = 0.02  # Minimum detectable effect (relative)
    mde_absolute: Optional[float] = None  # Minimum detectable effect (absolute)
    
    # Sequential testing parameters
    enable_sequential: bool = True
    max_looks: int = 10  # Maximum number of peeks
    alpha_spending: Literal["obf", "pocock", "linear"] = "obf"  # O'Brien-Fleming
    
    # CUPED parameters
    enable_cuped: bool = True
    pre_period_days: int = 30  # Days of pre-experiment data
    cuped_variables: List[str] = field(default_factory=lambda: ["prev_conversion_rate"])
    
    # Bayesian parameters
    enable_bayesian: bool = True
    prior_alpha: float = 1.0  # Beta prior alpha
    prior_beta: float = 1.0   # Beta prior beta
    credible_interval: float = 0.95
    rope_threshold: float = 0.001  # Region of practical equivalence
    
    # Business parameters
    daily_traffic: int = 10000
    conversion_value: float = 50.0  # Average value per conversion ($)
    implementation_cost: float = 10000.0  # One-time implementation cost ($)
    operational_cost_daily: float = 100.0  # Daily operational cost ($)
    
    # Guardrail metrics
    guardrail_metrics: List[str] = field(default_factory=lambda: [
        "bounce_rate", "page_load_time", "error_rate"
    ])
    guardrail_thresholds: Dict[str, float] = field(default_factory=lambda: {
        "bounce_rate": 0.05,  # Max allowed increase
        "page_load_time": 0.5,  # Max allowed increase (seconds)
        "error_rate": 0.001  # Max allowed increase
    })
    
    # Segmentation
    segments: List[str] = field(default_factory=lambda: [
        "device_type", "user_segment", "traffic_source"
    ])
    
    # Risk parameters
    risk_aversion: float = 1.0  # Risk multiplier (1.0 = neutral)
    confidence_threshold: float = 0.95  # For decision making
    
    # Decision framework
    auto_declare_winner: bool = True
    min_sample_size: int = 1000  # Minimum per variant
    min_runtime_days: int = 7  # Minimum days before declaring winner
    
    def to_dict(self) -> Dict:
        """Convert config to dictionary"""
        d = asdict(self)
        # Convert datetime to string
        d['start_date'] = self.start_date.isoformat()
        d['end_date'] = self.end_date.isoformat()
        return d
    
    def validate(self) -> bool:
        """Validate configuration parameters"""
        errors = []
        
        if not 0 < self.alpha < 1:
            errors.append("Alpha must be between 0 and 1")
        
        if not 0 < self.power < 1:
            errors.append("Power must be between 0 and 1")
        
        if self.start_date >= self.end_date:
            errors.append("Start date must be before end date")
        
        if self.daily_traffic <= 0:
            errors.append("Daily traffic must be positive")
        
        if errors:
            print("❌ Configuration errors:")
            for error in errors:
                print(f"  - {error}")
            return False
        
        print("✅ Configuration validated successfully")
        return True

# =============================================================================
# INITIALIZE CONFIGURATION
# =============================================================================

# Create default configuration
config = ExperimentConfig(
    experiment_name="Premium Feature Upsell Test",
    experiment_id="EXP-2024-PREMIUM-001",
    description="Testing new premium feature pricing and messaging",
    hypothesis="New pricing will increase premium conversions by 15%",
    
    # Statistical settings
    alpha=0.05,
    power=0.80,
    mde_relative=0.15,
    
    # Business parameters
    daily_traffic=50000,
    conversion_value=99.0,  # Premium subscription value
    implementation_cost=25000.0,
    operational_cost_daily=500.0,
    
    # Advanced features
    enable_sequential=True,
    enable_cuped=True,
    enable_bayesian=True
)

# Validate configuration
config.validate()

# Display configuration
print("\n📊 Experiment Configuration:")
print("-" * 50)
for key, value in config.to_dict().items():
    if not key.startswith('_'):
        print(f"{key:25s}: {value}")

## 🔧 2. Reusable Analysis Functions

In [ ]:
# =============================================================================
# CORE STATISTICAL FUNCTIONS
# =============================================================================

class ABTestAnalyzer:
    """Comprehensive A/B test analysis framework"""
    
    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.results = {}
    
    # -------------------------------------------------------------------------
    # Sample Size Calculations
    # -------------------------------------------------------------------------
    
    @staticmethod
    def calculate_sample_size(
        baseline_rate: float,
        mde: float,
        alpha: float = 0.05,
        power: float = 0.80,
        method: Literal["proportion", "continuous", "ratio"] = "proportion",
        variance: Optional[float] = None
    ) -> Dict[str, Any]:
        """Calculate sample size with multiple methods"""
        
        results = {}
        
        if method == "proportion":
            # Two-proportion z-test
            p1 = baseline_rate
            p2 = baseline_rate + mde
            
            # Pooled proportion
            p_pool = (p1 + p2) / 2
            
            # Z-scores
            z_alpha = stats.norm.ppf(1 - alpha/2)
            z_beta = stats.norm.ppf(power)
            
            # Sample size per group
            n = ((z_alpha + z_beta) ** 2 * 
                 (p1 * (1 - p1) + p2 * (1 - p2))) / (mde ** 2)
            
            results['n_per_group'] = int(np.ceil(n))
            results['total_n'] = 2 * results['n_per_group']
            results['method'] = 'Two-proportion z-test'
            
        elif method == "continuous":
            # Two-sample t-test
            if variance is None:
                variance = 1.0  # Assume unit variance
            
            effect_size = mde / np.sqrt(variance)
            n = tt_solve_power(effect_size=effect_size, 
                              alpha=alpha, 
                              power=power,
                              alternative='two-sided')
            
            results['n_per_group'] = int(np.ceil(n))
            results['total_n'] = 2 * results['n_per_group']
            results['method'] = 'Two-sample t-test'
            
        # Add runtime estimates
        if 'n_per_group' in results:
            results['days_needed'] = results['n_per_group'] / max(1000, 
                getattr(ABTestAnalyzer, '_daily_traffic', 10000))
            
        return results
    
    # -------------------------------------------------------------------------
    # SRM (Sample Ratio Mismatch) Check
    # -------------------------------------------------------------------------
    
    @staticmethod
    def check_srm(
        n_control: int,
        n_treatment: int,
        expected_ratio: float = 0.5,
        alpha: float = 0.05
    ) -> Dict[str, Any]:
        """Detailed SRM check with diagnostics"""
        
        total = n_control + n_treatment
        observed_ratio = n_treatment / total
        
        # Chi-square test
        expected_control = total * (1 - expected_ratio)
        expected_treatment = total * expected_ratio
        
        chi2_stat = ((n_control - expected_control) ** 2 / expected_control +
                    (n_treatment - expected_treatment) ** 2 / expected_treatment)
        
        p_value = 1 - stats.chi2.cdf(chi2_stat, df=1)
        
        # Binomial test as alternative
        binom_p = stats.binom_test(n_treatment, total, expected_ratio)
        
        # Severity assessment
        ratio_diff = abs(observed_ratio - expected_ratio)
        if ratio_diff < 0.01:
            severity = "Low"
        elif ratio_diff < 0.03:
            severity = "Medium"
        else:
            severity = "High"
        
        return {
            'has_srm': p_value < alpha,
            'p_value': p_value,
            'chi2_stat': chi2_stat,
            'binom_p_value': binom_p,
            'observed_ratio': observed_ratio,
            'expected_ratio': expected_ratio,
            'n_control': n_control,
            'n_treatment': n_treatment,
            'severity': severity,
            'recommendation': "Investigate assignment mechanism" if p_value < alpha else "No issues detected"
        }
    
    # -------------------------------------------------------------------------
    # Novelty Effect Detection
    # -------------------------------------------------------------------------
    
    @staticmethod
    def detect_novelty_effect(
        df: pd.DataFrame,
        date_col: str = 'date',
        conversion_col: str = 'converted',
        group_col: str = 'group',
        window_days: int = 3
    ) -> Dict[str, Any]:
        """Detect novelty effect in conversion rates over time"""
        
        # Calculate daily conversion rates
        daily_rates = df.groupby([date_col, group_col])[conversion_col].agg(['mean', 'count'])
        daily_rates = daily_rates.reset_index()
        
        results = {}
        
        for group in df[group_col].unique():
            group_data = daily_rates[daily_rates[group_col] == group].copy()
            group_data = group_data.sort_values(date_col)
            
            # Calculate rolling average
            group_data['rolling_mean'] = group_data['mean'].rolling(
                window=window_days, min_periods=1
            ).mean()
            
            # Fit linear regression to detect trend
            X = np.arange(len(group_data)).reshape(-1, 1)
            y = group_data['mean'].values
            
            model = LinearRegression()
            model.fit(X, y)
            
            slope = model.coef_[0]
            r2 = model.score(X, y)
            
            # Test for significant decline
            early_mean = group_data['mean'].iloc[:window_days].mean()
            late_mean = group_data['mean'].iloc[-window_days:].mean()
            
            decline_pct = (late_mean - early_mean) / early_mean * 100
            
            results[group] = {
                'slope': slope,
                'r2': r2,
                'early_mean': early_mean,
                'late_mean': late_mean,
                'decline_pct': decline_pct,
                'has_novelty': decline_pct < -10 and slope < 0,  # >10% decline
                'daily_data': group_data
            }
        
        # Overall assessment
        treatment_novelty = results.get('treatment', {}).get('has_novelty', False)
        
        return {
            'has_novelty_effect': treatment_novelty,
            'group_results': results,
            'recommendation': (
                "Consider extending test duration" if treatment_novelty 
                else "No significant novelty effect detected"
            )
        }
    
    # -------------------------------------------------------------------------
    # Automated Winner Declaration
    # -------------------------------------------------------------------------
    
    def declare_winner(
        self,
        results: Dict[str, Any],
        guardrail_checks: Optional[Dict[str, bool]] = None
    ) -> Dict[str, Any]:
        """Automated winner declaration with guardrails"""
        
        decision = {
            'winner': None,
            'confidence': 0,
            'can_ship': False,
            'reasons': [],
            'risks': []
        }
        
        # Check sample size
        if results.get('n_control', 0) < self.config.min_sample_size:
            decision['reasons'].append("Insufficient sample size")
            return decision
        
        # Check runtime
        runtime_days = (datetime.now() - self.config.start_date).days
        if runtime_days < self.config.min_runtime_days:
            decision['reasons'].append(f"Minimum runtime not met ({runtime_days}/{self.config.min_runtime_days} days)")
            return decision
        
        # Check statistical significance
        p_value = results.get('p_value', 1.0)
        if p_value >= self.config.alpha:
            decision['reasons'].append("Not statistically significant")
            decision['confidence'] = 1 - p_value
            return decision
        
        # Check practical significance
        lift = results.get('lift_absolute', 0)
        if abs(lift) < self.config.mde_absolute:
            decision['reasons'].append("Below minimum detectable effect")
            return decision
        
        # Check guardrails
        if guardrail_checks:
            failed_guardrails = [k for k, v in guardrail_checks.items() if not v]
            if failed_guardrails:
                decision['risks'].extend([f"Failed guardrail: {g}" for g in failed_guardrails])
                decision['can_ship'] = False
                return decision
        
        # Determine winner
        if lift > 0:
            decision['winner'] = 'treatment'
        else:
            decision['winner'] = 'control'
        
        decision['confidence'] = 1 - p_value
        decision['can_ship'] = True
        decision['reasons'].append(f"Clear winner: {decision['winner']}")
        
        return decision

# Initialize analyzer
analyzer = ABTestAnalyzer(config)

## 📈 3. Advanced Statistical Methods

In [ ]:
# =============================================================================
# SEQUENTIAL TESTING WITH ALPHA SPENDING
# =============================================================================

class SequentialTesting:
    """Sequential testing with proper alpha spending functions"""
    
    @staticmethod
    def obrien_fleming_bounds(information_fractions: np.ndarray, alpha: float = 0.05) -> np.ndarray:
        """O'Brien-Fleming boundaries"""
        z_alpha = stats.norm.ppf(1 - alpha/2)
        return z_alpha / np.sqrt(information_fractions)
    
    @staticmethod
    def pocock_bounds(n_looks: int, alpha: float = 0.05) -> np.ndarray:
        """Pocock boundaries"""
        # Approximation for common alpha levels
        pocock_constants = {
            2: 2.178, 3: 2.289, 4: 2.361, 5: 2.413,
            6: 2.453, 7: 2.486, 8: 2.512, 9: 2.535, 10: 2.555
        }
        
        if n_looks in pocock_constants and alpha == 0.05:
            return np.full(n_looks, pocock_constants[n_looks])
        else:
            # Fallback to O'Brien-Fleming
            fractions = np.linspace(1/n_looks, 1, n_looks)
            return SequentialTesting.obrien_fleming_bounds(fractions, alpha)
    
    @staticmethod
    def alpha_spending(
        t: float,
        alpha: float = 0.05,
        method: Literal["obf", "pocock", "linear"] = "obf"
    ) -> float:
        """Alpha spending function"""
        
        if method == "obf":
            # O'Brien-Fleming-like spending
            z_alpha = stats.norm.ppf(1 - alpha/2)
            return 2 * (1 - stats.norm.cdf(z_alpha / np.sqrt(t)))
        
        elif method == "pocock":
            # Pocock-like spending
            return alpha * np.log(1 + (np.e - 1) * t)
        
        else:  # linear
            return alpha * t
    
    @staticmethod
    def sequential_test(
        data: pd.DataFrame,
        n_looks: int,
        alpha: float = 0.05,
        spending: str = "obf"
    ) -> Dict[str, Any]:
        """Perform sequential test with alpha spending"""
        
        n_total = len(data)
        look_sizes = np.linspace(n_total // n_looks, n_total, n_looks, dtype=int)
        information_fractions = look_sizes / n_total
        
        # Calculate boundaries
        if spending == "obf":
            boundaries = SequentialTesting.obrien_fleming_bounds(information_fractions, alpha)
        elif spending == "pocock":
            boundaries = SequentialTesting.pocock_bounds(n_looks, alpha)
        else:
            boundaries = np.full(n_looks, stats.norm.ppf(1 - alpha/2))
        
        results = {
            'looks': [],
            'stopped_early': False,
            'stopping_look': None
        }
        
        for i, (size, boundary) in enumerate(zip(look_sizes, boundaries)):
            subset = data.iloc[:size]
            
            # Calculate test statistic
            control = subset[subset['group'] == 'control']['converted']
            treatment = subset[subset['group'] == 'treatment']['converted']
            
            n_c, n_t = len(control), len(treatment)
            x_c, x_t = control.sum(), treatment.sum()
            
            p_c, p_t = x_c / n_c, x_t / n_t
            p_pooled = (x_c + x_t) / (n_c + n_t)
            
            se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_c + 1/n_t))
            z_stat = (p_t - p_c) / se if se > 0 else 0
            
            look_result = {
                'look': i + 1,
                'n': size,
                'information_fraction': information_fractions[i],
                'z_stat': z_stat,
                'boundary': boundary,
                'p_value': 2 * (1 - stats.norm.cdf(abs(z_stat))),
                'reject_null': abs(z_stat) > boundary
            }
            
            results['looks'].append(look_result)
            
            if look_result['reject_null']:
                results['stopped_early'] = True
                results['stopping_look'] = i + 1
                break
        
        return results

# =============================================================================
# CUPED VARIANCE REDUCTION
# =============================================================================

class CUPED:
    """CUPED (Controlled-experiment Using Pre-Experiment Data) implementation"""
    
    def __init__(self, pre_metric: np.ndarray, post_metric: np.ndarray):
        """
        Args:
            pre_metric: Pre-experiment metric values
            post_metric: Post-experiment metric values
        """
        self.pre = np.asarray(pre_metric)
        self.post = np.asarray(post_metric)
        self.theta = None
        self.adjusted = None
        
    def fit(self) -> 'CUPED':
        """Calculate optimal theta and adjust metrics"""
        
        # Calculate covariance and variance
        cov = np.cov(self.post, self.pre)[0, 1]
        var_pre = np.var(self.pre)
        
        # Optimal theta
        self.theta = cov / var_pre if var_pre > 0 else 0
        
        # Adjusted metric
        self.adjusted = self.post - self.theta * (self.pre - np.mean(self.pre))
        
        return self
    
    def get_variance_reduction(self) -> float:
        """Calculate variance reduction percentage"""
        if self.adjusted is None:
            self.fit()
        
        var_original = np.var(self.post)
        var_adjusted = np.var(self.adjusted)
        
        return 1 - (var_adjusted / var_original) if var_original > 0 else 0
    
    def analyze(self, treatment: np.ndarray) -> Dict[str, Any]:
        """Analyze with and without CUPED adjustment"""
        
        if self.adjusted is None:
            self.fit()
        
        # Split by treatment
        control_mask = treatment == 0
        treatment_mask = treatment == 1
        
        # Original analysis
        orig_control = self.post[control_mask]
        orig_treatment = self.post[treatment_mask]
        
        orig_diff = np.mean(orig_treatment) - np.mean(orig_control)
        orig_se = np.sqrt(np.var(orig_treatment)/len(orig_treatment) + 
                         np.var(orig_control)/len(orig_control))
        
        # CUPED analysis
        adj_control = self.adjusted[control_mask]
        adj_treatment = self.adjusted[treatment_mask]
        
        adj_diff = np.mean(adj_treatment) - np.mean(adj_control)
        adj_se = np.sqrt(np.var(adj_treatment)/len(adj_treatment) + 
                        np.var(adj_control)/len(adj_control))
        
        return {
            'theta': self.theta,
            'variance_reduction': self.get_variance_reduction(),
            'original': {
                'difference': orig_diff,
                'se': orig_se,
                'ci_lower': orig_diff - 1.96 * orig_se,
                'ci_upper': orig_diff + 1.96 * orig_se,
                'p_value': 2 * (1 - stats.norm.cdf(abs(orig_diff / orig_se)))
            },
            'cuped': {
                'difference': adj_diff,
                'se': adj_se,
                'ci_lower': adj_diff - 1.96 * adj_se,
                'ci_upper': adj_diff + 1.96 * adj_se,
                'p_value': 2 * (1 - stats.norm.cdf(abs(adj_diff / adj_se)))
            },
            'power_gain': (orig_se / adj_se) ** 2 if adj_se > 0 else 1
        }

# =============================================================================
# BAYESIAN A/B TESTING
# =============================================================================

class BayesianAB:
    """Bayesian A/B testing with conjugate priors"""
    
    def __init__(
        self,
        prior_alpha: float = 1,
        prior_beta: float = 1,
        n_samples: int = 100000
    ):
        self.prior_alpha = prior_alpha
        self.prior_beta = prior_beta
        self.n_samples = n_samples
        
    def update_posterior(
        self,
        successes: int,
        failures: int
    ) -> Tuple[float, float]:
        """Update posterior parameters"""
        
        posterior_alpha = self.prior_alpha + successes
        posterior_beta = self.prior_beta + failures
        
        return posterior_alpha, posterior_beta
    
    def analyze(
        self,
        control_successes: int,
        control_total: int,
        treatment_successes: int,
        treatment_total: int,
        rope_threshold: float = 0.001
    ) -> Dict[str, Any]:
        """Perform Bayesian analysis"""
        
        # Update posteriors
        control_alpha, control_beta = self.update_posterior(
            control_successes,
            control_total - control_successes
        )
        
        treatment_alpha, treatment_beta = self.update_posterior(
            treatment_successes,
            treatment_total - treatment_successes
        )
        
        # Sample from posteriors
        control_samples = np.random.beta(control_alpha, control_beta, self.n_samples)
        treatment_samples = np.random.beta(treatment_alpha, treatment_beta, self.n_samples)
        
        # Calculate metrics
        diff_samples = treatment_samples - control_samples
        
        prob_treatment_better = np.mean(diff_samples > 0)
        prob_treatment_worse = np.mean(diff_samples < 0)
        prob_rope = np.mean(np.abs(diff_samples) < rope_threshold)
        
        # Expected lift
        expected_lift = np.mean(diff_samples)
        
        # Credible intervals
        ci_lower = np.percentile(diff_samples, 2.5)
        ci_upper = np.percentile(diff_samples, 97.5)
        
        # Risk analysis
        expected_loss_control = np.maximum(diff_samples, 0).mean()
        expected_loss_treatment = np.maximum(-diff_samples, 0).mean()
        
        return {
            'posterior_control': (control_alpha, control_beta),
            'posterior_treatment': (treatment_alpha, treatment_beta),
            'prob_treatment_better': prob_treatment_better,
            'prob_treatment_worse': prob_treatment_worse,
            'prob_rope': prob_rope,
            'expected_lift': expected_lift,
            'credible_interval': (ci_lower, ci_upper),
            'expected_loss_control': expected_loss_control,
            'expected_loss_treatment': expected_loss_treatment,
            'recommendation': self._get_recommendation(
                prob_treatment_better, expected_loss_treatment, expected_loss_control
            )
        }
    
    def _get_recommendation(
        self,
        prob_better: float,
        loss_treatment: float,
        loss_control: float
    ) -> str:
        """Generate recommendation based on results"""
        
        if prob_better > 0.95 and loss_treatment < loss_control * 0.5:
            return "Strong evidence to choose treatment"
        elif prob_better > 0.80:
            return "Moderate evidence favoring treatment"
        elif prob_better < 0.20:
            return "Evidence favoring control"
        else:
            return "Insufficient evidence - continue testing"
    
    def prior_sensitivity_analysis(
        self,
        control_data: Dict,
        treatment_data: Dict,
        prior_range: List[Tuple[float, float]]
    ) -> pd.DataFrame:
        """Analyze sensitivity to prior choice"""
        
        results = []
        
        for alpha, beta in prior_range:
            self.prior_alpha = alpha
            self.prior_beta = beta
            
            analysis = self.analyze(
                control_data['successes'],
                control_data['total'],
                treatment_data['successes'],
                treatment_data['total']
            )
            
            results.append({
                'prior_alpha': alpha,
                'prior_beta': beta,
                'prob_treatment_better': analysis['prob_treatment_better'],
                'expected_lift': analysis['expected_lift']
            })
        
        return pd.DataFrame(results)

# =============================================================================
# DIFFERENCE-IN-DIFFERENCES
# =============================================================================

class DifferenceInDifferences:
    """DiD for observational causal inference"""
    
    def __init__(self):
        self.model = None
        self.results = None
    
    def fit(
        self,
        df: pd.DataFrame,
        outcome: str,
        treatment: str,
        time: str,
        covariates: Optional[List[str]] = None
    ) -> 'DifferenceInDifferences':
        """Fit DiD model"""
        
        # Create interaction term
        df['treat_x_time'] = df[treatment] * df[time]
        
        # Build formula
        formula = f"{outcome} ~ {treatment} + {time} + treat_x_time"
        
        if covariates:
            formula += " + " + " + ".join(covariates)
        
        # Fit model
        self.model = sm.OLS.from_formula(formula, data=df).fit(
            cov_type='HC1'  # Heteroskedasticity-robust SE
        )
        
        self.results = {
            'did_estimate': self.model.params['treat_x_time'],
            'did_se': self.model.bse['treat_x_time'],
            'did_pvalue': self.model.pvalues['treat_x_time'],
            'did_ci': self.model.conf_int().loc['treat_x_time'].values,
            'model_summary': self.model.summary()
        }
        
        return self
    
    def parallel_trends_test(
        self,
        df: pd.DataFrame,
        outcome: str,
        treatment: str,
        time_periods: List[str]
    ) -> Dict[str, Any]:
        """Test parallel trends assumption"""
        
        # Pre-treatment periods only
        pre_df = df[df['post'] == 0].copy()
        
        trends = {}
        for period in time_periods:
            control = pre_df[pre_df[treatment] == 0].groupby(period)[outcome].mean()
            treated = pre_df[pre_df[treatment] == 1].groupby(period)[outcome].mean()
            trends[period] = {
                'control': control,
                'treated': treated,
                'difference': treated - control
            }
        
        # Test if trends are parallel (interaction should be non-significant)
        formula = f"{outcome} ~ {treatment} * C({time_periods[0]})"
        test_model = sm.OLS.from_formula(formula, data=pre_df).fit()
        
        interaction_p = test_model.pvalues[f"{treatment}:C({time_periods[0]})[T.1]"]
        
        return {
            'trends': trends,
            'parallel_trends_pvalue': interaction_p,
            'assumption_holds': interaction_p > 0.05,
            'interpretation': (
                "Parallel trends assumption holds" if interaction_p > 0.05
                else "Warning: Parallel trends assumption may be violated"
            )
        }

## 📊 4. Enhanced Visualizations

In [ ]:
# =============================================================================
# INTERACTIVE PLOTLY VISUALIZATIONS
# =============================================================================

class ABTestVisualizer:
    """Enhanced visualization suite for A/B tests"""
    
    @staticmethod
    def plot_cumulative_conversion(
        df: pd.DataFrame,
        date_col: str = 'date',
        group_col: str = 'group',
        conversion_col: str = 'converted'
    ) -> go.Figure:
        """Interactive cumulative conversion rate plot with confidence bands"""
        
        # Calculate cumulative metrics
        results = []
        
        for group in df[group_col].unique():
            group_df = df[df[group_col] == group].sort_values(date_col)
            
            cumsum = group_df[conversion_col].cumsum()
            cumcount = np.arange(1, len(group_df) + 1)
            cum_rate = cumsum / cumcount
            
            # Calculate confidence intervals
            se = np.sqrt(cum_rate * (1 - cum_rate) / cumcount)
            ci_lower = cum_rate - 1.96 * se
            ci_upper = cum_rate + 1.96 * se
            
            results.append({
                'group': group,
                'dates': group_df[date_col].values,
                'cum_rate': cum_rate,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper
            })
        
        # Create plotly figure
        fig = go.Figure()
        
        colors = ['blue', 'red', 'green', 'orange']
        
        for i, res in enumerate(results):
            color = colors[i % len(colors)]
            
            # Main line
            fig.add_trace(go.Scatter(
                x=res['dates'],
                y=res['cum_rate'],
                mode='lines',
                name=f"{res['group']} (rate)",
                line=dict(color=color, width=2)
            ))
            
            # Confidence band
            fig.add_trace(go.Scatter(
                x=np.concatenate([res['dates'], res['dates'][::-1]]),
                y=np.concatenate([res['ci_upper'], res['ci_lower'][::-1]]),
                fill='toself',
                fillcolor=f'rgba({color[4:-1]}, 0.2)',
                line=dict(color='rgba(255,255,255,0)'),
                showlegend=False,
                hoverinfo='skip'
            ))
        
        fig.update_layout(
            title='Cumulative Conversion Rates with 95% Confidence Bands',
            xaxis_title='Date',
            yaxis_title='Cumulative Conversion Rate',
            hovermode='x unified',
            template='plotly_white'
        )
        
        return fig
    
    @staticmethod
    def plot_decision_tree(
        test_results: Dict[str, Any]
    ) -> go.Figure:
        """Interactive decision tree for test interpretation"""
        
        # Define decision nodes
        nodes = [
            "Start",
            "SRM Check",
            "Sample Size",
            "Statistical Sig",
            "Practical Sig",
            "Guardrails",
            "Ship",
            "Don't Ship",
            "Investigate"
        ]
        
        # Define edges based on results
        edges = []
        edge_labels = []
        
        # SRM check
        if test_results.get('srm_passed', True):
            edges.append(("Start", "Sample Size"))
            edge_labels.append("SRM Pass")
        else:
            edges.append(("Start", "Investigate"))
            edge_labels.append("SRM Fail")
        
        # Continue building tree based on results...
        
        # Create Sankey diagram
        fig = go.Figure(data=[go.Sankey(
            node=dict(
                pad=15,
                thickness=20,
                line=dict(color="black", width=0.5),
                label=nodes,
                color=["blue", "green", "green", "orange", "orange", 
                      "orange", "green", "red", "yellow"]
            ),
            link=dict(
                source=[nodes.index(e[0]) for e in edges],
                target=[nodes.index(e[1]) for e in edges],
                value=[1] * len(edges),
                label=edge_labels
            )
        )])
        
        fig.update_layout(
            title="A/B Test Decision Tree",
            font_size=12
        )
        
        return fig
    
    @staticmethod
    def plot_power_heatmap(
        baseline_rates: np.ndarray,
        effect_sizes: np.ndarray,
        sample_size: int,
        alpha: float = 0.05
    ) -> go.Figure:
        """Power analysis heatmap for different scenarios"""
        
        # Calculate power for each combination
        power_matrix = np.zeros((len(baseline_rates), len(effect_sizes)))
        
        for i, baseline in enumerate(baseline_rates):
            for j, effect in enumerate(effect_sizes):
                # Calculate power using normal approximation
                p1 = baseline
                p2 = baseline * (1 + effect)
                
                pooled_p = (p1 + p2) / 2
                se = np.sqrt(2 * pooled_p * (1 - pooled_p) / sample_size)
                
                z_alpha = stats.norm.ppf(1 - alpha/2)
                ncp = abs(p2 - p1) / se  # Non-centrality parameter
                
                power = 1 - stats.norm.cdf(z_alpha - ncp) + stats.norm.cdf(-z_alpha - ncp)
                power_matrix[i, j] = power
        
        # Create heatmap
        fig = go.Figure(data=go.Heatmap(
            z=power_matrix,
            x=[f"{e:.1%}" for e in effect_sizes],
            y=[f"{b:.1%}" for b in baseline_rates],
            colorscale='RdYlGn',
            text=power_matrix.round(3),
            texttemplate="%{text}",
            textfont={"size": 10},
            colorbar=dict(title="Power")
        ))
        
        fig.update_layout(
            title=f"Statistical Power Heatmap (n={sample_size:,} per group, α={alpha})",
            xaxis_title="Relative Effect Size",
            yaxis_title="Baseline Conversion Rate",
            width=800,
            height=600
        )
        
        return fig
    
    @staticmethod
    def plot_sequential_boundaries(
        sequential_results: Dict[str, Any]
    ) -> go.Figure:
        """Plot sequential testing boundaries and test statistics"""
        
        looks_data = pd.DataFrame(sequential_results['looks'])
        
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=('Test Statistic vs Boundaries', 'P-values Over Time'),
            vertical_spacing=0.15
        )
        
        # Plot 1: Test statistics and boundaries
        fig.add_trace(
            go.Scatter(
                x=looks_data['information_fraction'],
                y=looks_data['z_stat'],
                mode='lines+markers',
                name='Test Statistic',
                line=dict(color='blue', width=2)
            ),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=looks_data['information_fraction'],
                y=looks_data['boundary'],
                mode='lines',
                name='Upper Boundary',
                line=dict(color='red', dash='dash')
            ),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=looks_data['information_fraction'],
                y=-looks_data['boundary'],
                mode='lines',
                name='Lower Boundary',
                line=dict(color='red', dash='dash')
            ),
            row=1, col=1
        )
        
        # Plot 2: P-values
        fig.add_trace(
            go.Scatter(
                x=looks_data['information_fraction'],
                y=looks_data['p_value'],
                mode='lines+markers',
                name='P-value',
                line=dict(color='green', width=2)
            ),
            row=2, col=1
        )
        
        # Add significance threshold
        fig.add_hline(y=0.05, line_dash="dash", line_color="gray", 
                     annotation_text="α=0.05", row=2, col=1)
        
        fig.update_xaxes(title_text="Information Fraction", row=2, col=1)
        fig.update_yaxes(title_text="Z-statistic", row=1, col=1)
        fig.update_yaxes(title_text="P-value", row=2, col=1)
        
        fig.update_layout(
            height=700,
            title_text="Sequential Testing Results",
            showlegend=True
        )
        
        return fig

# Initialize visualizer
visualizer = ABTestVisualizer()

## 💰 5. Business Impact Analysis

In [ ]:
# =============================================================================
# BUSINESS IMPACT CALCULATOR
# =============================================================================

class BusinessImpactAnalyzer:
    """Comprehensive business impact and ROI analysis"""
    
    def __init__(self, config: ExperimentConfig):
        self.config = config
    
    def calculate_revenue_impact(
        self,
        lift: float,
        ci_lower: float,
        ci_upper: float,
        time_horizon_days: int = 365
    ) -> Dict[str, Any]:
        """Calculate revenue impact with confidence intervals"""
        
        daily_conversions_base = self.config.daily_traffic * 0.1  # Assume 10% baseline
        
        # Point estimate
        daily_lift_conversions = daily_conversions_base * lift
        daily_revenue_lift = daily_lift_conversions * self.config.conversion_value
        total_revenue_lift = daily_revenue_lift * time_horizon_days
        
        # Confidence intervals
        daily_lift_lower = daily_conversions_base * ci_lower
        daily_lift_upper = daily_conversions_base * ci_upper
        
        revenue_lower = daily_lift_lower * self.config.conversion_value * time_horizon_days
        revenue_upper = daily_lift_upper * self.config.conversion_value * time_horizon_days
        
        return {
            'daily_revenue_lift': daily_revenue_lift,
            'total_revenue_lift': total_revenue_lift,
            'revenue_ci_lower': revenue_lower,
            'revenue_ci_upper': revenue_upper,
            'confidence_interval': (revenue_lower, revenue_upper),
            'time_horizon_days': time_horizon_days
        }
    
    def cost_benefit_analysis(
        self,
        revenue_impact: Dict[str, Any],
        include_opportunity_cost: bool = True
    ) -> Dict[str, Any]:
        """Comprehensive cost-benefit analysis"""
        
        # Costs
        implementation_cost = self.config.implementation_cost
        operational_cost = self.config.operational_cost_daily * revenue_impact['time_horizon_days']
        
        # Opportunity cost (time spent not shipping winner)
        test_duration = (self.config.end_date - self.config.start_date).days
        opportunity_cost = 0
        if include_opportunity_cost:
            opportunity_cost = revenue_impact['daily_revenue_lift'] * test_duration * 0.5
        
        total_cost = implementation_cost + operational_cost + opportunity_cost
        
        # Benefits
        total_benefit = revenue_impact['total_revenue_lift']
        benefit_lower = revenue_impact['revenue_ci_lower']
        benefit_upper = revenue_impact['revenue_ci_upper']
        
        # ROI calculations
        roi = (total_benefit - total_cost) / total_cost if total_cost > 0 else float('inf')
        roi_lower = (benefit_lower - total_cost) / total_cost if total_cost > 0 else float('inf')
        roi_upper = (benefit_upper - total_cost) / total_cost if total_cost > 0 else float('inf')
        
        # Payback period
        daily_net_benefit = revenue_impact['daily_revenue_lift'] - self.config.operational_cost_daily
        payback_days = implementation_cost / daily_net_benefit if daily_net_benefit > 0 else float('inf')
        
        return {
            'total_cost': total_cost,
            'total_benefit': total_benefit,
            'net_benefit': total_benefit - total_cost,
            'roi': roi,
            'roi_confidence_interval': (roi_lower, roi_upper),
            'payback_period_days': payback_days,
            'break_even_date': self.config.start_date + timedelta(days=int(payback_days)),
            'cost_breakdown': {
                'implementation': implementation_cost,
                'operational': operational_cost,
                'opportunity': opportunity_cost
            }
        }
    
    def sensitivity_analysis(
        self,
        base_lift: float,
        parameter_ranges: Dict[str, List[float]]
    ) -> pd.DataFrame:
        """Sensitivity analysis for key assumptions"""
        
        results = []
        
        # Base case
        base_revenue = self.calculate_revenue_impact(
            base_lift, base_lift * 0.8, base_lift * 1.2
        )
        base_cba = self.cost_benefit_analysis(base_revenue)
        
        # Vary each parameter
        for param, values in parameter_ranges.items():
            for value in values:
                # Create modified config
                temp_config = self.config
                
                if param == 'conversion_value':
                    original = temp_config.conversion_value
                    temp_config.conversion_value = value
                elif param == 'daily_traffic':
                    original = temp_config.daily_traffic
                    temp_config.daily_traffic = value
                elif param == 'lift':
                    original = base_lift
                    base_lift = value
                
                # Recalculate
                revenue = self.calculate_revenue_impact(
                    base_lift, base_lift * 0.8, base_lift * 1.2
                )
                cba = self.cost_benefit_analysis(revenue)
                
                results.append({
                    'parameter': param,
                    'value': value,
                    'net_benefit': cba['net_benefit'],
                    'roi': cba['roi'],
                    'change_from_base': (cba['net_benefit'] - base_cba['net_benefit']) / 
                                      base_cba['net_benefit'] if base_cba['net_benefit'] != 0 else 0
                })
                
                # Reset
                if param == 'conversion_value':
                    temp_config.conversion_value = original
                elif param == 'daily_traffic':
                    temp_config.daily_traffic = original
                elif param == 'lift':
                    base_lift = original
        
        return pd.DataFrame(results)
    
    def decision_framework(
        self,
        test_results: Dict[str, Any],
        revenue_impact: Dict[str, Any],
        cost_benefit: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Multi-metric decision framework"""
        
        decision_matrix = {
            'statistical_significance': test_results.get('p_value', 1.0) < self.config.alpha,
            'practical_significance': abs(test_results.get('lift', 0)) > self.config.mde_relative,
            'positive_roi': cost_benefit['roi'] > 0.2,  # 20% ROI threshold
            'acceptable_payback': cost_benefit['payback_period_days'] < 180,
            'guardrails_pass': test_results.get('guardrails_pass', True),
            'sufficient_power': test_results.get('achieved_power', 0) > 0.7
        }
        
        # Calculate decision score
        weights = {
            'statistical_significance': 0.25,
            'practical_significance': 0.20,
            'positive_roi': 0.25,
            'acceptable_payback': 0.10,
            'guardrails_pass': 0.15,
            'sufficient_power': 0.05
        }
        
        decision_score = sum(
            weights[k] * (1 if v else 0) 
            for k, v in decision_matrix.items()
        )
        
        # Make recommendation
        if decision_score >= 0.8:
            recommendation = "STRONG SHIP"
            action = "Implement immediately"
        elif decision_score >= 0.6:
            recommendation = "SHIP"
            action = "Implement with monitoring"
        elif decision_score >= 0.4:
            recommendation = "CONDITIONAL"
            action = "Gather more data or iterate"
        else:
            recommendation = "DON'T SHIP"
            action = "Reject or significantly revise"
        
        return {
            'decision_matrix': decision_matrix,
            'decision_score': decision_score,
            'recommendation': recommendation,
            'action': action,
            'confidence_level': min(decision_score, test_results.get('confidence', 0))
        }

# Initialize business analyzer
business_analyzer = BusinessImpactAnalyzer(config)

## 🚀 6. Complete Analysis Pipeline

In [ ]:
# =============================================================================
# COMPLETE A/B TEST ANALYSIS PIPELINE
# =============================================================================

def run_complete_ab_analysis(
    df: pd.DataFrame,
    config: ExperimentConfig,
    pre_experiment_df: Optional[pd.DataFrame] = None
) -> Dict[str, Any]:
    """
    Run complete A/B test analysis pipeline
    
    Args:
        df: Experiment data
        config: Experiment configuration
        pre_experiment_df: Pre-experiment data for CUPED
    
    Returns:
        Comprehensive results dictionary
    """
    
    print("🔬 Starting Comprehensive A/B Test Analysis")
    print("=" * 60)
    
    results = {}
    
    # -------------------------------------------------------------------------
    # 1. Data Quality Checks
    # -------------------------------------------------------------------------
    print("\n📊 Step 1: Data Quality Checks")
    
    # SRM check
    n_control = len(df[df['group'] == 'control'])
    n_treatment = len(df[df['group'] == 'treatment'])
    
    srm_results = ABTestAnalyzer.check_srm(n_control, n_treatment)
    results['srm'] = srm_results
    
    print(f"  ✓ SRM Check: {'PASSED' if not srm_results['has_srm'] else 'FAILED'}")
    print(f"    - Control: {n_control:,}")
    print(f"    - Treatment: {n_treatment:,}")
    print(f"    - P-value: {srm_results['p_value']:.4f}")
    
    # -------------------------------------------------------------------------
    # 2. Primary Analysis
    # -------------------------------------------------------------------------
    print("\n📈 Step 2: Primary Statistical Analysis")
    
    # Calculate conversion rates
    control_data = df[df['group'] == 'control']
    treatment_data = df[df['group'] == 'treatment']
    
    control_conversions = control_data['converted'].sum()
    treatment_conversions = treatment_data['converted'].sum()
    
    control_rate = control_conversions / n_control
    treatment_rate = treatment_conversions / n_treatment
    
    # Z-test
    z_stat, p_value = proportions_ztest(
        [control_conversions, treatment_conversions],
        [n_control, n_treatment]
    )
    
    # Confidence interval
    ci_lower, ci_upper = proportion_confint(
        treatment_conversions - control_conversions,
        n_treatment + n_control,
        method='normal'
    )
    
    lift_absolute = treatment_rate - control_rate
    lift_relative = lift_absolute / control_rate if control_rate > 0 else 0
    
    results['primary'] = {
        'control_rate': control_rate,
        'treatment_rate': treatment_rate,
        'lift_absolute': lift_absolute,
        'lift_relative': lift_relative,
        'z_statistic': z_stat,
        'p_value': p_value,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'significant': p_value < config.alpha
    }
    
    print(f"  ✓ Control Rate: {control_rate:.4f}")
    print(f"  ✓ Treatment Rate: {treatment_rate:.4f}")
    print(f"  ✓ Lift: {lift_relative:.2%} (p={p_value:.4f})")
    
    # -------------------------------------------------------------------------
    # 3. Sequential Testing
    # -------------------------------------------------------------------------
    if config.enable_sequential:
        print("\n🔄 Step 3: Sequential Testing")
        
        seq_results = SequentialTesting.sequential_test(
            df, 
            n_looks=min(config.max_looks, 10),
            alpha=config.alpha,
            spending=config.alpha_spending
        )
        
        results['sequential'] = seq_results
        
        print(f"  ✓ Stopped early: {seq_results['stopped_early']}")
        if seq_results['stopped_early']:
            print(f"  ✓ Stopping look: {seq_results['stopping_look']}")
    
    # -------------------------------------------------------------------------
    # 4. CUPED Analysis
    # -------------------------------------------------------------------------
    if config.enable_cuped and pre_experiment_df is not None:
        print("\n🎯 Step 4: CUPED Variance Reduction")
        
        # Mock pre-experiment data (in practice, use actual data)
        pre_metric = np.random.beta(2, 8, len(df))
        post_metric = df['converted'].values
        treatment = (df['group'] == 'treatment').astype(int).values
        
        cuped = CUPED(pre_metric, post_metric)
        cuped_results = cuped.analyze(treatment)
        
        results['cuped'] = cuped_results
        
        print(f"  ✓ Variance reduction: {cuped_results['variance_reduction']:.2%}")
        print(f"  ✓ Power gain: {cuped_results['power_gain']:.2f}x")
    
    # -------------------------------------------------------------------------
    # 5. Bayesian Analysis
    # -------------------------------------------------------------------------
    if config.enable_bayesian:
        print("\n🎲 Step 5: Bayesian Analysis")
        
        bayesian = BayesianAB(
            prior_alpha=config.prior_alpha,
            prior_beta=config.prior_beta
        )
        
        bayes_results = bayesian.analyze(
            control_conversions, n_control,
            treatment_conversions, n_treatment,
            rope_threshold=config.rope_threshold
        )
        
        results['bayesian'] = bayes_results
        
        print(f"  ✓ P(Treatment > Control): {bayes_results['prob_treatment_better']:.2%}")
        print(f"  ✓ Expected lift: {bayes_results['expected_lift']:.4f}")
        print(f"  ✓ {bayes_results['recommendation']}")
    
    # -------------------------------------------------------------------------
    # 6. Novelty Effect Detection
    # -------------------------------------------------------------------------
    print("\n🆕 Step 6: Novelty Effect Detection")
    
    if 'date' in df.columns:
        novelty_results = ABTestAnalyzer.detect_novelty_effect(df)
        results['novelty'] = novelty_results
        
        print(f"  ✓ Novelty effect detected: {novelty_results['has_novelty_effect']}")
    
    # -------------------------------------------------------------------------
    # 7. Business Impact
    # -------------------------------------------------------------------------
    print("\n💰 Step 7: Business Impact Analysis")
    
    revenue_impact = business_analyzer.calculate_revenue_impact(
        lift_relative,
        ci_lower,
        ci_upper,
        time_horizon_days=365
    )
    
    cost_benefit = business_analyzer.cost_benefit_analysis(revenue_impact)
    
    results['business_impact'] = {
        'revenue': revenue_impact,
        'cost_benefit': cost_benefit
    }
    
    print(f"  ✓ Expected annual revenue lift: ${revenue_impact['total_revenue_lift']:,.0f}")
    print(f"  ✓ ROI: {cost_benefit['roi']:.1%}")
    print(f"  ✓ Payback period: {cost_benefit['payback_period_days']:.0f} days")
    
    # -------------------------------------------------------------------------
    # 8. Final Decision
    # -------------------------------------------------------------------------
    print("\n🎯 Step 8: Decision Framework")
    
    decision = business_analyzer.decision_framework(
        results['primary'],
        revenue_impact,
        cost_benefit
    )
    
    results['decision'] = decision
    
    print(f"  ✓ Decision Score: {decision['decision_score']:.2f}")
    print(f"  ✓ Recommendation: {decision['recommendation']}")
    print(f"  ✓ Action: {decision['action']}")
    
    print("\n" + "=" * 60)
    print("✅ Analysis Complete!")
    
    return results

# =============================================================================
# DEMO: Generate Sample Data and Run Analysis
# =============================================================================

# Generate sample data
np.random.seed(42)

n_samples = 50000
dates = pd.date_range(start='2024-01-01', periods=30, freq='D')

sample_data = []
for date in dates:
    daily_samples = n_samples // len(dates)
    
    # Control group
    control = pd.DataFrame({
        'user_id': np.arange(len(sample_data), len(sample_data) + daily_samples // 2),
        'group': 'control',
        'date': date,
        'converted': np.random.binomial(1, 0.10, daily_samples // 2)  # 10% baseline
    })
    
    # Treatment group with 15% lift
    treatment = pd.DataFrame({
        'user_id': np.arange(len(sample_data) + daily_samples // 2, 
                           len(sample_data) + daily_samples),
        'group': 'treatment',
        'date': date,
        'converted': np.random.binomial(1, 0.115, daily_samples // 2)  # 15% relative lift
    })
    
    sample_data.append(pd.concat([control, treatment]))

df_sample = pd.concat(sample_data, ignore_index=True)

print("📊 Sample Dataset Created")
print(f"  - Total samples: {len(df_sample):,}")
print(f"  - Date range: {df_sample['date'].min()} to {df_sample['date'].max()}")
print(f"  - Groups: {df_sample['group'].unique()}")
print()

# Run complete analysis
results = run_complete_ab_analysis(df_sample, config)

## 📊 7. Results Dashboard

In [ ]:
# =============================================================================
# INTERACTIVE RESULTS DASHBOARD
# =============================================================================

# Create comprehensive dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Conversion Rates by Group',
        'Cumulative Lift Over Time', 
        'Sequential Testing Boundaries',
        'Bayesian Posterior Distributions',
        'Business Impact Projection',
        'Decision Matrix'
    ),
    specs=[
        [{'type': 'bar'}, {'type': 'scatter'}],
        [{'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'waterfall'}, {'type': 'indicator'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.15
)

# Plot 1: Conversion Rates
fig.add_trace(
    go.Bar(
        x=['Control', 'Treatment'],
        y=[results['primary']['control_rate'], results['primary']['treatment_rate']],
        text=[f"{r:.2%}" for r in [results['primary']['control_rate'], 
                                   results['primary']['treatment_rate']]],
        textposition='auto',
        marker_color=['lightblue', 'lightgreen']
    ),
    row=1, col=1
)

# Plot 2: Cumulative Conversion Over Time
daily_cum = df_sample.groupby(['date', 'group'])['converted'].agg(['sum', 'count'])
daily_cum['rate'] = daily_cum['sum'].cumsum() / daily_cum['count'].cumsum()
daily_cum = daily_cum.reset_index()

for group in ['control', 'treatment']:
    group_data = daily_cum[daily_cum['group'] == group]
    fig.add_trace(
        go.Scatter(
            x=group_data['date'],
            y=group_data['rate'],
            mode='lines',
            name=group.capitalize(),
            line=dict(width=2)
        ),
        row=1, col=2
    )

# Update layout
fig.update_layout(
    height=900,
    title_text="A/B Test Results Dashboard",
    showlegend=True,
    template='plotly_white'
)

fig.show()

# Print summary report
print("\n" + "="*60)
print("📋 EXECUTIVE SUMMARY")
print("="*60)
print(f"""
Experiment: {config.experiment_name}
Duration: {(config.end_date - config.start_date).days} days
Total Sample: {len(df_sample):,}

RESULTS:
• Lift: {results['primary']['lift_relative']:.2%} (95% CI: [{results['primary']['ci_lower']:.2%}, {results['primary']['ci_upper']:.2%}])
• Statistical Significance: p-value = {results['primary']['p_value']:.4f}
• Practical Significance: {'YES' if abs(results['primary']['lift_relative']) > config.mde_relative else 'NO'}
• Expected Annual Revenue: ${results['business_impact']['revenue']['total_revenue_lift']:,.0f}
• ROI: {results['business_impact']['cost_benefit']['roi']:.1%}

RECOMMENDATION: {results['decision']['recommendation']}
ACTION: {results['decision']['action']}
""")

print("\n✅ Analysis complete! Results exported and visualized.")